In [142]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns   
import holidays

import ipywidgets as widgets
from ipywidgets import interact, fixed
import calendar
import random 
import warnings
warnings.filterwarnings("ignore")

In [143]:
df = pd.read_csv("../data/data_processed.csv")

df.head()

,sku,shipped_date,qty
0,017SAD,2021-01-01,0.0
1,017SAD,2021-01-03,75.0
2,017SAD,2021-01-05,0.0
3,017SAD,2021-01-07,0.0
4,017SAD,2021-01-09,0.0


In [144]:
df.dtypes

sku                 str
shipped_date        str
qty             float64
dtype: object

In [145]:
df["shipped_date"] = pd.to_datetime(df["shipped_date"])
df = df.sort_values(["sku", "shipped_date"]).reset_index(drop=True)

1. Time features

In [146]:
df["month"] = df["shipped_date"].dt.month
df["day_of_week"] = df["shipped_date"].dt.dayofweek

2. Lag features

In [147]:
df["lag_1"] = (df.groupby("sku")["qty"].shift(1))

df["lag_2"] = (df.groupby("sku")["qty"].shift(2))

df["lag_3"] = (df.groupby("sku")["qty"].shift(3))
df["lag_4"] = (df.groupby("sku")["qty"].shift(4))
df["lag_5"] = (df.groupby("sku")["qty"].shift(5))
df["lag_6"] = (df.groupby("sku")["qty"].shift(6))
df["lag_7"] = (df.groupby("sku")["qty"].shift(7))

3. Rolling features

In [148]:
df["rolling_mean_3"] = (
    df.groupby("sku")["qty"]
      .transform(
          lambda x: x.shift(1).rolling(3).mean()
      )
)
df["rolling_mean_5"] = (
    df.groupby("sku")["qty"]
      .transform(
          lambda x: x.shift(1).rolling(5).mean()
      )
)

df["rolling_mean_7"] = (
    df.groupby("sku")["qty"]
      .transform(
          lambda x: x.shift(1).rolling(7).mean()
      )
)

In [149]:
df["trend_7"] = df["lag_1"] - df["lag_7"]

In [150]:
df["growth_1"] = (
    (df["lag_1"] - df["lag_2"])
    / (df["lag_2"] + 1)
)

In [151]:
df["deviation_from_mean"] = (
    df["lag_1"] - df["rolling_mean_3"]
)

In [152]:
df[[
        "shipped_date",
        "sku",
        "qty",
        "lag_1",
        "lag_2",
        "lag_3",
        "lag_4",
        "lag_5",
        "lag_6",
        "lag_7",
        "rolling_mean_7",
        "rolling_mean_5",
        "rolling_mean_3",
        "trend_7",
        "growth_1",
        "deviation_from_mean"
    ]
].head()

,shipped_date,sku,qty,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,rolling_mean_7,rolling_mean_5,rolling_mean_3,trend_7,growth_1,deviation_from_mean
0,2021-01-01,017SAD,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-03,017SAD,75.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-05,017SAD,0.0,75.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.000000,NaN
3,2021-01-07,017SAD,0.0,0.0,75.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,25.0,NaN,-0.986842,-25.0
4,2021-01-09,017SAD,0.0,0.0,0.0,75.0,0.0,NaN,NaN,NaN,NaN,NaN,25.0,NaN,0.000000,-25.0


In [153]:
#drop NaN
df = df.dropna(subset=["lag_1","lag_2","lag_3","lag_4","lag_5","lag_6","lag_7","rolling_mean_3","rolling_mean_5","rolling_mean_7","trend_7","growth_1"]
               ).reset_index(drop=True)

In [154]:
data_engineered = df.copy()

data_engineered.to_csv(
    "../data/data_engineered.csv",
    index=False
)